# 16장 CNN과 LeNet — 위치가 달라도 같은 모양

**3단계 · 다시 만들기** — 『법률가를 위한 인공지능 코딩 자습서』 실습 노트북

[![Colab에서 열기](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch16_3_build.ipynb)

이 노트북에는 **주석만 남긴 코드**가 들어 있습니다. 책과 1단계 노트북을 덮고, 주석을 실마리 삼아 코드를 되살려 보세요. 아래 표의 「다시 만들기」 과제가 이 노트북의 목표입니다.

- 코드 셀 첫 줄의 `%%writefile 파일이름` 은 "이 셀의 내용을 그 파일로 저장하라"는 뜻입니다. 채운 뒤 셀을 실행해 저장하고, 확인 셀로 돌려 봅니다.
- 막히면 오류 메시지를 먼저 읽고(부록 A), 그래도 안 되면 1단계 노트북에서 그 부분만 확인하세요.

| 단계 | 할 일 | 노트북 |
|---|---|---|
| 1. 따라 하기 | `p01`~`p04` 와 `lenet.py` 를 치고 실행. `p02` 의 숫자를 종이에 먼저 계산한 뒤 실행 | [ch16_1_follow.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch16_1_follow.ipynb) |
| 2. 주석 달기 | `LeNet.forward` 각 줄 뒤에 그 줄을 지난 shape을 주석으로 | [ch16_2_comment.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch16_2_comment.ipynb) |
| 3. 다시 만들기 | 주석만 남긴 파일에서 `LeNet` 복원. 그다음 `conv` 를 세 층으로 늘린 나만의 변형을 만들어 `p03` 의 표에 한 줄 추가 | [ch16_3_build.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch16_3_build.ipynb) **(지금 이 노트북)** |

In [ ]:
# 준비 셀 — 맨 먼저 한 번 실행하세요. 실습 코드와 데이터를 받아 옵니다.
import os, sys
if "google.colab" in sys.modules:                       # Colab 에서 열었을 때
    ROOT = "/content/AI-Coding-for-Lawyers"
    if not os.path.isdir(ROOT):
        !git clone -q --depth 1 https://github.com/BowonKwon/AI-Coding-for-Lawyers.git {ROOT}
else:                                                   # 내 컴퓨터에서 열었을 때 (저장소 폴더 안)
    ROOT = os.getcwd()
    while not os.path.isdir(os.path.join(ROOT, "notebooks")) and os.path.dirname(ROOT) != ROOT:
        ROOT = os.path.dirname(ROOT)
os.environ.setdefault("JUCOS_DATA", os.path.join(ROOT, "data"))   # 데이터를 두는 폴더
os.chdir(os.path.join(ROOT, "code", "16"))                      # 이 장의 코드 폴더로 이동
if not os.path.isdir(os.path.join(os.environ["JUCOS_DATA"], "shapes")):   # 15장의 도형 데이터가 없으면 만든다
    !python ../15/make_shapes.py
print("준비 끝. 작업 폴더: code/16")

## `lenet.py` — 대상: `LeNet`

In [ ]:
%%writefile lenet.py
# 16.2 LeNet 계열 CNN: 합성곱 두 번 → 펴기 → 선형 두 번. 14장 networks 패키지에 넣을 수 있는 형태.
import torch
import torch.nn as nn
import torch.nn.functional as F


class LeNet(nn.Module):
    """입력 [B, in_ch, 28, 28] (MNIST) 또는 [B, in_ch, 32, 32] (도형·CIFAR) 를 받는다."""

    def __init__(self, in_ch=1, num_classes=10, img_size=28):
        # 28 → 24 (32 → 28)
        # 12 → 8  (14 → 10)  (풀링 뒤)
        # 풀링 두 번 뒤의 한 변: 28→4, 32→5
        raise NotImplementedError("여기를 채우세요")

    def forward(self, x):
        # 합성곱 → 활성화 → 풀링
        # [B, 16*side*side]
        raise NotImplementedError("여기를 채우세요")


class LeNetWide(LeNet):
    """구조 변경 실험 1: 채널을 두 배로. 부모의 forward 는 그대로 쓴다."""

    def __init__(self, in_ch=1, num_classes=10, img_size=28):
        super().__init__(in_ch, num_classes, img_size)
        side = ((img_size - 4) // 2 - 4) // 2
        self.conv1 = nn.Conv2d(in_ch, 12, kernel_size=5)
        self.conv2 = nn.Conv2d(12, 32, kernel_size=5)
        self.fc1 = nn.Linear(32 * side * side, 120)


class LeNetDeep(nn.Module):
    """구조 변경 실험 2: 합성곱을 한 층 더, padding 으로 크기를 유지하며 쌓기."""

    def __init__(self, in_ch=1, num_classes=10, img_size=28):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, 16, 3, padding=1)          # 크기 유지
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        self.conv3 = nn.Conv2d(32, 64, 3, padding=1)
        side = img_size // 8                                      # 풀링 세 번: 28→3, 32→4
        self.fc = nn.Linear(64 * side * side, num_classes)

    def forward(self, x):
        x = F.max_pool2d(F.relu(self.conv1(x)), 2)
        x = F.max_pool2d(F.relu(self.conv2(x)), 2)
        x = F.max_pool2d(F.relu(self.conv3(x)), 2)
        return self.fc(x.flatten(1))


if __name__ == "__main__":
    for cls in (LeNet, LeNetWide, LeNetDeep):
        m = cls()
        n = sum(p.numel() for p in m.parameters())
        print(f"{cls.__name__:<10} 파라미터 {n:>8,}  [64,1,28,28] → {tuple(m(torch.zeros(64, 1, 28, 28)).shape)}")
    m = LeNet(in_ch=3, num_classes=3, img_size=32)
    print(f"LeNet(3ch,32px) 파라미터 {sum(p.numel() for p in m.parameters()):,}  [8,3,32,32] → {tuple(m(torch.zeros(8, 3, 32, 32)).shape)}")

### 확인

In [ ]:
!python lenet.py

### 확인

In [ ]:
!python p02_shape_trace.py

### 원래대로 되돌리기

파일을 책의 원래 코드로 되돌리고 싶으면 아래 셀을 실행하세요. (Colab 에서 받은 저장소 기준입니다.)

In [ ]:
!git checkout -- lenet.py

## 그다음

`conv` 를 세 층으로 늘린 나만의 변형을 만들어 `p03_compare.py` 의 표에 한 줄 추가하세요.

In [ ]:
# 여기에 이어서 작성하세요